In [ ]:
cssrs = pd.read_csv('data/cssrs_standardized.csv')
mindset = pd.read_csv('data/mindset_standardized.csv')
umd = pd.read_csv('data/umd_standardized.csv')

cssrs['dataset'] = 'cssrs'
mindset['dataset'] = 'mindset'
umd['dataset'] = 'umd'

df = pd.concat([cssrs, mindset, umd], ignore_index=True)
df['text'] = df['sentence'].fillna('')
df['binary_label'] = df['label'].apply(lambda x: 1 if x in ['suicidal', 'at_risk', 'has_condition'] else 0)

print(f"Total samples: {len(df)}")
print(f"Datasets: {df['dataset'].value_counts().to_dict()}")

In [ ]:
# ================================================================
# GENDER DETECTION PATTERNS --- 7 tiers, ordered by confidence
# ================================================================

# --- Tier 1: AGE + GENDER combos (Reddit convention, highest conf) ---
AGE_GENDER = [
    # (25F)  [30M]  (25 female)
    re.compile(r'[\(\[]\s*(\d{1,3})\s*(m|f|nb|male|female|nonbinary|enby)\s*[\)\]]', re.I),
    # F(25)  male(30)
    re.compile(r'\b(m|f|male|female|nb|nonbinary|enby)\s*[\(\[]\s*(\d{1,3})\s*[\)\]]', re.I),
    # 25/F  30/M  25/female
    re.compile(r'\b(\d{1,3})\s*/\s*(m|f|nb|male|female|nonbinary|enby)\b', re.I),
    # 25f  30m  (adjacent, no separator)
    re.compile(r'(?<![.\d/])\b(\d{1,3})\s*(m|f|nb|male|female|nonbinary|enby)\b', re.I),
    # 25 year old female / 30yo male
    re.compile(
        r'\b(\d{1,3})\s*(?:yo|y/?o|years?\s*old|yrs?\s*old)\s+'
        r'(male|female|man|woman|boy|girl|guy|gal|lady|dude|m|f)\b', re.I),
    # I'm a 25 year old female
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:a\s+)?(\d{1,3})\s*"
        r"(?:yo|y/?o|years?\s*old|yrs?\s*old)\s+"
        r"(male|female|man|woman|boy|girl|guy|gal|lady|dude|m|f)\b", re.I),
]

# --- Tier 2: EXPLICIT self-identification ---------------------------------
SELF_ID = [
    # I am [a] girl/guy/man/woman/male/female/...
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:a\s+)?"
        r"(girl|boy|man|woman|male|female|guy|gal|lady|dude|gentleman"
        r"|nonbinary|nb|enby)\b", re.I),
    # I am [a] [adjective] girl/guy/...
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:a\s+)?"
        r"(?:young|old|older|teenage|teen|grown|middle[\s-]aged|elderly"
        r"|single|married|divorced|widowed|straight|gay|bisexual|bi)\s+"
        r"(girl|boy|man|woman|male|female|guy|gal|lady|dude)\b", re.I),
    # as a [gender] / as a young woman
    re.compile(
        r'\bas\s+a\s+'
        r'(girl|boy|man|woman|male|female|guy|gal|lady|dude|gentleman'
        r'|young\s+woman|young\s+man|young\s+lady|young\s+guy'
        r'|grown\s+man|grown\s+woman|grown\s+lady'
        r'|teenage\s+girl|teenage\s+boy|teen\s+girl|teen\s+boy)\b', re.I),
    # being [a] [gender]
    re.compile(
        r'\b(?:being|become|becoming)\s+(?:a\s+)?'
        r'(girl|boy|man|woman|male|female|guy|gal|lady|dude)\b', re.I),
    # [gender] here
    re.compile(
        r'\b(girl|boy|man|woman|male|female|guy|gal|lady|dude|gentleman)\s+here\b', re.I),
    # biologically [male/female]
    re.compile(r'\bbiologically\s+(male|female)\b', re.I),
    # I'm considered / seen as / perceived as [gender]
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:considered|seen\s+as|perceived\s+as|identified\s+as)\s+"
        r"(?:a\s+)?(male|female|man|woman)\b", re.I),
]

# --- Tier 3: TRANS / NB identity ------------------------------------------
TRANS_NB = [
    re.compile(
        r'\b(trans\s*woman|trans\s*man|transwoman|transman'
        r'|trans\s*girl|trans\s*boy|trans\s*guy|trans\s*gal'
        r'|trans\s*lady|trans\s*male|trans\s*female'
        r'|detrans\s*woman|detrans\s*man)\b', re.I),
    re.compile(r'\b(ftm|mtf|f2m|m2f)\b', re.I),
    re.compile(r'\b(amab|afab)\b', re.I),
    re.compile(r'\bassigned\s+(male|female)\s+at\s+birth\b', re.I),
]

# --- Tier 4: PARENTAL / FAMILY self-roles ---------------------------------
PARENTAL = [
    # I am [a] mom/dad/...
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:a\s+)?"
        r"(mom|mother|mommy|mama|mum|mummy|dad|father|daddy|papa)\b", re.I),
    # as a single mom / stay-at-home dad / ...
    re.compile(
        r'\bas\s+a\s+'
        r'(mom|mother|mommy|mama|mum|mummy|dad|father|daddy|papa'
        r'|single\s+mom|single\s+dad|single\s+mother|single\s+father'
        r'|new\s+mom|new\s+dad|new\s+mother|new\s+father'
        r'|stay[\s-]at[\s-]home\s+mom|stay[\s-]at[\s-]home\s+dad)\b', re.I),
    # Standalone compound roles
    re.compile(
        r'\b(single\s+mom|single\s+dad|single\s+mother|single\s+father'
        r'|new\s+mom|new\s+dad|sahm|sahd'
        r'|stay[\s-]at[\s-]home\s+mom|stay[\s-]at[\s-]home\s+dad)\b', re.I),
]

# --- Tier 5: FELLOW [group] membership ------------------------------------
FELLOW = [
    re.compile(
        r'\bfellow\s+(guys?|girls?|ladies|men|women|boys?|dudes?'
        r'|gals?|gentlemen|brothers?|sisters?'
        r'|moms?|dads?|mothers?|fathers?)\b', re.I),
]

# --- Tier 6: GENDER-SPECIFIC EXPERIENCES (implies female) -----------------
GENDERED_EXP = [
    re.compile(r"\b(?:i\s+am|i'm|im|i\s+was|when\s+i\s+was|being)\s+pregnant\b", re.I),
    re.compile(r'\bmy\s+(?:menstruation|menstrual\s+cycle|pms|period\s+cramps?)\b', re.I),
    re.compile(r'\bmy\s+period(?!\s+of\b)\b', re.I),
    re.compile(r"\b(?:i\s+am|i'm|im)\s+(?:currently\s+)?(?:breastfeeding|nursing)\b", re.I),
    re.compile(r'\bmy\s+pregnancy\b', re.I),
    re.compile(r"\b(?:i\s+)?(?:gave|giving)\s+birth\b", re.I),
    re.compile(r"\b(?:i\s+have|i've|my)\s+postpartum\b", re.I),
]

# --- Tier 7: PARENTHETICAL gender markers ---------------------------------
PAREN_GENDER = [
    # I (F) / me (M)
    re.compile(r'\b(?:i|me|myself)\s*[\(\[]\s*\d*\s*(f|m|nb|male|female|nonbinary)\s*[\)\]]', re.I),
    # Sentence-initial or after punctuation: (F) (M)
    re.compile(r'(?:^|[.!?,;:\n])\s*[\(\[]\s*(f|m|nb|male|female|nonbinary)\s*[\)\]]', re.I | re.M),
]

ALL_TIERS = [AGE_GENDER, SELF_ID, TRANS_NB, PARENTAL, FELLOW, GENDERED_EXP, PAREN_GENDER]
print(f"Pattern tiers: 7  |  Total patterns: {sum(len(t) for t in ALL_TIERS)}")

In [ ]:
def normalize_gender(value):
    """Map raw gender captures to canonical labels."""
    if not value:
        return None
    v = re.sub(r'\s+', ' ', value.strip().lower())

    mapping = {
        # ---- Male ----
        "m": "male", "male": "male", "man": "male", "boy": "male",
        "guy": "male", "dude": "male", "gentleman": "male",
        "dad": "male", "father": "male", "daddy": "male", "papa": "male",
        "single dad": "male", "single father": "male",
        "new dad": "male", "new father": "male", "sahd": "male",
        "stay-at-home dad": "male", "stay at home dad": "male",
        "guys": "male", "men": "male", "boys": "male",
        "dudes": "male", "brothers": "male", "brother": "male",
        "gentlemen": "male", "dads": "male", "fathers": "male",
        "young man": "male", "young guy": "male", "grown man": "male",
        "teenage boy": "male", "teen boy": "male",

        # ---- Female ----
        "f": "female", "female": "female", "woman": "female", "girl": "female",
        "gal": "female", "lady": "female",
        "mom": "female", "mother": "female", "mommy": "female",
        "mama": "female", "mum": "female", "mummy": "female",
        "single mom": "female", "single mother": "female",
        "new mom": "female", "new mother": "female", "sahm": "female",
        "stay-at-home mom": "female", "stay at home mom": "female",
        "girls": "female", "women": "female", "ladies": "female",
        "gals": "female", "sisters": "female", "sister": "female",
        "moms": "female", "mothers": "female", "mamas": "female",
        "young woman": "female", "young lady": "female",
        "grown woman": "female", "grown lady": "female",
        "teenage girl": "female", "teen girl": "female",

        # ---- Nonbinary ----
        "nb": "nonbinary", "nonbinary": "nonbinary", "enby": "nonbinary",

        # ---- Trans ----
        "trans man": "trans_male", "transman": "trans_male",
        "trans boy": "trans_male", "trans guy": "trans_male",
        "trans male": "trans_male",
        "ftm": "trans_male", "f2m": "trans_male",
        "trans woman": "trans_female", "transwoman": "trans_female",
        "trans girl": "trans_female", "trans gal": "trans_female",
        "trans lady": "trans_female", "trans female": "trans_female",
        "mtf": "trans_female", "m2f": "trans_female",
        "detrans man": "detrans_male", "detrans woman": "detrans_female",

        # ---- Assigned at birth ----
        "amab": "amab", "afab": "afab",
    }
    return mapping.get(v, v)


def is_valid_age(age):
    """Check if extracted age is plausible."""
    try:
        return 10 <= int(age) <= 100
    except (ValueError, TypeError):
        return False

In [ ]:
TIER_NAMES = ["age_gender", "self_id", "trans_nb", "parental",
              "fellow", "gendered_exp", "paren_gender"]
TIER_CONF  = ["high", "high", "high", "medium",
              "medium", "medium", "medium"]

def extract_gender(text):
    """
    Extract gender self-disclosure from text using 7-tier pattern matching.
    Returns dict with: gender, matched_excerpt, pattern_tier, confidence.
    """
    if pd.isna(text):
        text = ""
    text = str(text)

    # Normalise smart quotes / apostrophes
    text = text.replace('\u2019', "'").replace('\u2018', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')

    # Strip quoted text and Reddit block-quotes (avoid matching others' words)
    clean = re.sub(r'"[^"]*"', ' ', text)
    clean = re.sub(r"(?m)^[ \t]*>.*$", ' ', clean)

    result = {"gender": None, "matched_excerpt": None,
              "pattern_tier": None, "confidence": None}

    for tier_pats, tier_name, tier_conf in zip(ALL_TIERS, TIER_NAMES, TIER_CONF):
        for pat in tier_pats:
            m = pat.search(clean)
            if m:
                # Identify which capture group holds the gender token
                gender_raw = None
                for g in m.groups():
                    if g and not g.isdigit():
                        gender_raw = g
                        break

                # Tier 6 (gendered experiences) always implies female
                if tier_name == "gendered_exp":
                    gender_raw = "female"

                if gender_raw:
                    result.update(
                        gender=normalize_gender(gender_raw),
                        matched_excerpt=m.group(0).strip(),
                        pattern_tier=tier_name,
                        confidence=tier_conf,
                    )
                    return result

    return result

In [ ]:
extracted = df['text'].progress_apply(extract_gender).apply(pd.Series)
df_annotated = pd.concat([df.reset_index(drop=True), extracted], axis=1)

n_gender = df_annotated['gender'].notna().sum()
print(f"Rows: {len(df_annotated):,}")
print(f"Gender found: {n_gender:,}  ({n_gender/len(df_annotated)*100:.2f}%)")

In [ ]:
print("=" * 60)
print("GENDER DISTRIBUTION")
print("=" * 60)
print(df_annotated['gender'].value_counts(dropna=False))

print(f"\n{'=' * 60}\nBY DATASET\n{'=' * 60}")
print(pd.crosstab(df_annotated['dataset'], df_annotated['gender'], margins=True))

print(f"\n{'=' * 60}\nBY LABEL\n{'=' * 60}")
print(pd.crosstab(df_annotated['binary_label'], df_annotated['gender'], margins=True))

print(f"\n{'=' * 60}\nBY PATTERN TIER\n{'=' * 60}")
print(df_annotated['pattern_tier'].value_counts(dropna=False))

print(f"\n{'=' * 60}\nBY CONFIDENCE\n{'=' * 60}")
print(df_annotated['confidence'].value_counts(dropna=False))

In [ ]:
sample = df_annotated.loc[
    df_annotated['gender'].notna(),
    ['text', 'dataset', 'label', 'gender', 'matched_excerpt', 'pattern_tier', 'confidence']
]
print(f"Total gender-annotated: {len(sample)}")
for tier in TIER_NAMES:
    subset = sample[sample['pattern_tier'] == tier]
    if len(subset) > 0:
        print(f"\n--- {tier} ({len(subset)} matches) ---")
        display(subset[['matched_excerpt', 'gender', 'dataset']].head(5))

In [ ]:
df_annotated.to_pickle('data/gender_annotated.pkl')

for ds in ['cssrs', 'mindset', 'umd']:
    subset = df_annotated[df_annotated['dataset'] == ds]
    subset.to_csv(f'data/{ds}_gender_annotated.csv', index=False)
    g_count = subset['gender'].notna().sum()
    print(f"{ds}: {len(subset):,} rows, {g_count:,} gender annotations ({g_count/len(subset)*100:.2f}%)")

print(f"\nTotal: {df_annotated['gender'].notna().sum():,} / {len(df_annotated):,} "
      f"({df_annotated['gender'].notna().mean()*100:.2f}%)")
print("Saved: data/gender_annotated.pkl + per-dataset CSVs")